# Baseline Model: Linear Regression

## Introduction
In this notebook, we will create a **baseline model** to predict building energy consumption. A baseline model is a simple model that serves as a starting point. It gives us a reference performance score that we try to beat with more complex models later.

We will use **Linear Regression**, which is a fundamental algorithm in machine learning. It assumes a linear relationship between the input features (like temperature, square feet) and the target variable (energy consumption).

## Steps
1. **Load Data**: Import the processed training and validation datasets.
2. **Prepare Data**: Select relevant features and handle any remaining missing values.
3. **Train Model**: Fit a Linear Regression model on the training data.
4. **Evaluate**: Measure performance using RMSE (Root Mean Squared Error) and RMSLE (Root Mean Squared Logarithmic Error).
5. **Visualize**: Plot predictions against actual values to understand where the model makes mistakes.

In [ ]:
# Import necessary libraries
import pandas as pd  # For data manipulation
import numpy as np   # For numerical operations
import matplotlib.pyplot as plt # For plotting
import seaborn as sns # For nicer plots
from sklearn.linear_model import LinearRegression # The model we will use
from sklearn.metrics import mean_squared_error # Metric to evaluate performance
import os

# Set plot style for better readability
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## 1. Load Data
We will load the parquet files we created in the previous phase. Parquet is a column-oriented storage format that is very efficient for large datasets.

In [ ]:
# Define paths to our data
DATA_DIR = "../../data/processed"
TRAIN_PATH = os.path.join(DATA_DIR, "train_split.parquet")
VAL_PATH = os.path.join(DATA_DIR, "val_split.parquet")

# Load the datasets
print("Loading data...")
train_df = pd.read_parquet(TRAIN_PATH)
val_df = pd.read_parquet(VAL_PATH)

print(f"Training set shape: {train_df.shape}")
print(f"Validation set shape: {val_df.shape}")

# Display the first few rows to verify
train_df.head()

## 2. Feature Selection
We need to decide which columns to use as inputs (features) for our model. We should exclude columns that are not useful for prediction (like IDs) or columns that *are* the target itself.

**Target Variable**: `meter_reading`

**Features to Drop**:
- `timestamp`: The raw date/time is hard for a linear model to use directly. We have already extracted features like `hour`, `day`, `month`.
- `meter_reading`: This is what we want to predict!
- `site_id`, `building_id`: These are high-cardinality categorical variables. For a simple baseline, we might skip them or use them if they are encoded. Let's check our columns.

In [ ]:
# List all columns to see what we have
print(train_df.columns.tolist())

In [ ]:
# Define our features and target
target = "meter_reading"

# Let's select numerical and encoded features. 
# We drop 'timestamp' and the target variable.
# We also drop 'meter' if we want to predict for all meters generally, 
# but usually, meter type is very important. Let's keep it.

features = [
    'square_feet', 'year_built', 'floor_count', # Building metadata
    'air_temperature', 'cloud_coverage', 'dew_temperature', # Weather
    'precip_depth_1_hr', 'sea_level_pressure', 'wind_direction', 'wind_speed', # More weather
    'hour', 'day', 'weekday', 'month', # Time features
    'primary_use', 'meter' # Categorical features (already encoded/numeric)
]

# Check if all these columns exist in our dataframe
missing_cols = [col for col in features if col not in train_df.columns]
if missing_cols:
    print(f"Warning: The following columns are missing: {missing_cols}")
    # Remove missing columns from features list
    features = [col for col in features if col in train_df.columns]

print(f"Selected {len(features)} features: {features}")

### Handling Missing Values (Imputation)
Linear Regression cannot handle missing values (`NaN`). Although we did some preprocessing, let's double-check if any NaNs remain and fill them with a simple strategy (like the mean) just to be safe.

In [ ]:
# Check for missing values in features
print("Missing values in training features:")
print(train_df[features].isnull().sum()[train_df[features].isnull().sum() > 0])

# Simple imputation: Fill NaNs with the mean of the column
# Note: In a real production pipeline, we should fit the imputer on train and transform val/test.
# For this baseline, simple fillna is acceptable.
for col in features:
    if train_df[col].isnull().sum() > 0:
        mean_val = train_df[col].mean()
        train_df[col] = train_df[col].fillna(mean_val)
        val_df[col] = val_df[col].fillna(mean_val)
        print(f"Filled missing values in {col} with {mean_val:.2f}")

## 3. Train Model
Now we initialize the Linear Regression model and fit it to our training data.

In [ ]:
# Prepare X (features) and y (target)
X_train = train_df[features]
y_train = train_df[target]

X_val = val_df[features]
y_val = val_df[target]

# Initialize the model
model = LinearRegression()

# Train the model
print("Training Linear Regression model...")
model.fit(X_train, y_train)
print("Training complete.")

## 4. Evaluate Model
We will predict on the validation set and calculate the error.

**RMSE (Root Mean Squared Error)**: Standard metric for regression. Lower is better.
**RMSLE (Root Mean Squared Logarithmic Error)**: Often used in energy competitions because it cares more about the *ratio* of error than the absolute difference. It's less sensitive to outliers.

In [ ]:
# Make predictions on validation set
y_pred = model.predict(X_val)

# Linear Regression can sometimes predict negative values, which is impossible for energy.
# Let's clip predictions to be at least 0.
y_pred = np.maximum(y_pred, 0)

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(f"Validation RMSE: {rmse:.4f}")

# Calculate RMSLE
# We use np.log1p to calculate log(1 + x) safely
rmsle = np.sqrt(mean_squared_error(np.log1p(y_val), np.log1p(y_pred)))
print(f"Validation RMSLE: {rmsle:.4f}")

## 5. Visualize Results
Let's plot a small sample of our predictions vs the actual values to see how well the model is tracking the trends.

In [ ]:
# Create a dataframe for comparison
results_df = pd.DataFrame({
    'Actual': y_val,
    'Predicted': y_pred
})

# Plot a subset of data (first 200 points) to avoid clutter
plt.figure(figsize=(15, 6))
subset = results_df.iloc[:200]
plt.plot(subset['Actual'].values, label='Actual', alpha=0.7)
plt.plot(subset['Predicted'].values, label='Predicted', alpha=0.7, linestyle='--')
plt.title("Actual vs Predicted Energy Consumption (First 200 Validation Samples)")
plt.xlabel("Sample Index")
plt.ylabel("Meter Reading")
plt.legend()
plt.show()

## 6. Key Insights & Observations

### Performance Metrics
- **Validation RMSE**: ~10,595
- **Validation RMSLE**: ~4.47

### Interpretation
1.  **High Error**: An RMSLE of 4.47 is quite high for this dataset (top Kaggle solutions are under 1.10). This confirms that a simple Linear Regression model is **underfitting** the data.
2.  **Non-Linearity**: Energy consumption often has complex, non-linear relationships with temperature and time (e.g., HVAC systems work harder at both very hot and very cold temperatures). Linear models struggle to capture these U-shaped patterns.
3.  **Visual Discrepancy**: The plot likely shows that the model predicts the "average" well but fails to capture the peaks and valleys of daily energy usage.

### Next Steps
- **Tree-Based Models**: We will move to **LightGBM** or **XGBoost**. These models naturally handle non-linearities and interactions between features (e.g., how the effect of temperature changes depending on the hour of the day).
- **Feature Engineering**: We can add more complex features, but switching to a better model class is the highest priority intervention right now.